# FedDiff-MSA: Federated Diffusion-based Multimodal Sentiment Analysis

This notebook runs the full experiment suite on Google Colab with GPU.

**Setup:** Runtime → Change runtime type → **T4 GPU**

| Experiment | Est. Time (T4) | Paper Table |
|-----------|---------------|-------------|
| Main results | ~8h | Table 3 |
| Recovery quality | ~50min | Table 4 |
| Ablation study | ~10h | Table 5 |
| Privacy-utility | ~5h | Figure 6 |
| MIA | ~2.5h | Table 6 |
| Scalability | ~3h | Table 7 |
| Figures | ~5h | Fig 2-7 |

> **Note:** 12-hour session limit. Run experiments separately if needed. Results are saved to CSV/JSON and persist across sessions via Google Drive mount.

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Mount Google Drive (to save results persistently across sessions)
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")

In [ ]:
# Clone the repository
%cd /content
!rm -rf FedDiff-MSA
!git clone https://github.com/yjyjss/FedDiff-MSA.git
%cd FedDiff-MSA

# Pull LFS files (the .npy data files)
!git lfs install
!git lfs pull

# Verify data files
import os
data_dir = 'data/features'
for f in ['mosei_text.npy', 'mosei_audio.npy', 'mosei_visual.npy', 'mosei_labels.npy']:
    path = os.path.join(data_dir, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {f}: {size_mb:.1f} MB")
    else:
        print(f"  {f}: MISSING!")

In [ ]:
# Install dependencies
!pip install -r requirements.txt
!pip install scikit-learn matplotlib
print("Dependencies installed.")

In [ ]:
# Quick test to verify code works
!python run_experiments.py --test --exp main 2>&1 | tail -20

## 2. Configure Output Directory

Results will be saved to Google Drive so they persist across Colab sessions.

In [ ]:
import os

# Save results to Google Drive (persistent across sessions)
OUTPUT_DIR = '/content/drive/MyDrive/FedDiff-MSA-Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Also create a symlink so the code writes there
!rm -rf outputs
!ln -s {OUTPUT_DIR} outputs

print(f"Results will be saved to: {OUTPUT_DIR}")
print(f"Symlink: outputs -> {OUTPUT_DIR}")

## 3. Run Experiments

Run each experiment separately. Each cell is independent — if your session disconnects, just re-run the setup cells above and continue with the next experiment.

### 3.1 Main Experiment → Table 3
9 methods × Setting C. ~8 hours on T4.

If 12h limit is a concern, run in 3 batches (3 methods each).

In [ ]:
# Full main experiment (all 9 methods, ~8h)
!python run_experiments.py --dataset mosei --setting C --device cuda --exp main 2>&1 | tee outputs/log_main.txt

### 3.2 Recovery Quality → Table 4
~50 minutes on T4.

In [ ]:
!python run_experiments.py --dataset mosei --setting C --device cuda --exp recovery 2>&1 | tee outputs/log_recovery.txt

### 3.3 Ablation Study → Table 5
11 variants. ~10 hours on T4.

> If session times out, results from completed variants are already saved. Re-running will overwrite — consider running subsets by editing the `ablation_configs` dict in `run_experiments.py`.

In [ ]:
!python run_experiments.py --dataset mosei --setting C --device cuda --exp ablation 2>&1 | tee outputs/log_ablation.txt

### 3.4 Privacy-Utility Trade-off → Figure 6
Layered DP vs Uniform DP across epsilon. ~5 hours on T4.

In [ ]:
!python run_experiments.py --dataset mosei --setting C --device cuda --exp privacy 2>&1 | tee outputs/log_privacy.txt

### 3.5 MIA Success Rates → Table 6
~2.5 hours on T4.

In [ ]:
!python run_experiments.py --dataset mosei --setting C --device cuda --exp mia 2>&1 | tee outputs/log_mia.txt

### 3.6 Client Scalability → Table 7
~3 hours on T4.

In [ ]:
!python run_experiments.py --dataset mosei --setting C --device cuda --exp scalability 2>&1 | tee outputs/log_scalability.txt

## 4. Generate Figures

Run after experiments are complete. Each figure reads from CSV/JSON results.

In [ ]:
# Generate all figures (reads from existing experiment results)
!python generate_figures.py --device cuda 2>&1 | tee outputs/log_figures.txt

Or generate individual figures:

In [ ]:
# Individual figures
# !python generate_figures.py --device cuda --fig convergence
# !python generate_figures.py --device cuda --fig tsne
# !python generate_figures.py --device cuda --fig noniid
# !python generate_figures.py --device cuda --fig missing_ratio
# !python generate_figures.py --device cuda --fig privacy_utility
# !python generate_figures.py --device cuda --fig contribution

## 5. View Results

In [ ]:
# List all output files
import os
output_dir = '/content/drive/MyDrive/FedDiff-MSA-Results'

print("=== CSV Table Files ===")
for f in sorted(os.listdir(output_dir)):
    if f.endswith('.csv'):
        size = os.path.getsize(os.path.join(output_dir, f)) / 1024
        print(f"  {f:45s} {size:.1f} KB")

print("\n=== JSON Detail Files ===")
for f in sorted(os.listdir(output_dir)):
    if f.endswith('.json'):
        size = os.path.getsize(os.path.join(output_dir, f)) / 1024
        print(f"  {f:45s} {size:.1f} KB")

print("\n=== Figures ===")
fig_dir = os.path.join(output_dir, 'figures')
if os.path.exists(fig_dir):
    for f in sorted(os.listdir(fig_dir)):
        if f.endswith('.pdf') or f.endswith('.png'):
            size = os.path.getsize(os.path.join(fig_dir, f)) / 1024
            print(f"  {f:45s} {size:.1f} KB")

In [ ]:
# Display a sample CSV result
import pandas as pd

csv_files = [f for f in os.listdir(output_dir) if f.endswith('.csv') and f.startswith('table_')]
if csv_files:
    print(f"Available tables: {csv_files}")
    print()
    # Show the first one
    df = pd.read_csv(os.path.join(output_dir, csv_files[0]), comment='#')
    print(f"=== {csv_files[0]} ===")
    print(df.to_string())

In [ ]:
# Display figures
from IPython.display import Image, display
import os

fig_dir = os.path.join(output_dir, 'figures')
if os.path.exists(fig_dir):
    png_files = sorted([f for f in os.listdir(fig_dir) if f.endswith('.png')])
    for f in png_files:
        print(f"\n=== {f} ===")
        display(Image(os.path.join(fig_dir, f)))
else:
    print("No figures found. Run the figure generation cell first.")

## 6. Download Results

Results are already in your Google Drive (`FedDiff-MSA-Results/`). You can also download specific files:

In [ ]:
# Download all CSV results as a zip
!cd /content/drive/MyDrive/FedDiff-MSA-Results && zip -j /content/results.zip *.csv *.json *.md 2>/dev/null

from google.colab import files
files.download('/content/results.zip')

## 7. Push Results Back to GitHub (Optional)

Upload experiment results to a new `results` branch on GitHub.

In [ ]:
# Configure git and push results
# Replace YOUR_TOKEN with a GitHub Personal Access Token
# GITHUB_TOKEN = 'YOUR_TOKEN'
# !git config user.name "yjyjss"
# !git config user.email "yjyjss@users.noreply.github.com"
# !git remote set-url origin https://{GITHUB_TOKEN}@github.com/yjyjss/FedDiff-MSA.git
# !git checkout -b results
# !cp /content/drive/MyDrive/FedDiff-MSA-Results/*.csv .
# !cp /content/drive/MyDrive/FedDiff-MSA-Results/*.json .
# !git add *.csv *.json
# !git commit -m "Add experiment results from Colab (T4 GPU)"
# !git push origin results
print("Uncomment the code above and add your token to push results to GitHub.")